# Task 04 — EDA & Statistical Insights

Explore the cleaned synthetic retail dataset from Task 03. The notebook calculates descriptive statistics, category/city/segment/monthly summaries, correlations, visualizations, three hypothesis tests, and five data-driven findings.

**Transparency:** this is synthetic practice data, not real company data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

DATA_PATH = Path('../RabTech_Task03_Deliverables/clean_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('clean_dataset.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['order_date'])
print('Rows and columns:', df.shape)
print('Date range:', df['order_date'].min().date(), 'to', df['order_date'].max().date())
display(df.head())


## 1. Descriptive statistics

In [ ]:
display(df.describe(include='all').T)
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Missing values by column:')
display(df.isna().sum().rename('missing_count').to_frame())


## 2. Grouped analysis

In [ ]:
category_summary = df.groupby('category', as_index=False).agg(order_rows=('order_id','count'), units_sold=('quantity','sum'), gross_sales=('gross_sales','sum'), net_sales=('net_sales','sum'), avg_net_sales=('net_sales','mean'), avg_discount_pct=('discount_pct','mean')).sort_values('net_sales', ascending=False)
city_summary = df.groupby('city', as_index=False).agg(order_rows=('order_id','count'), net_sales=('net_sales','sum'), avg_net_sales=('net_sales','mean')).sort_values('net_sales', ascending=False)
segment_summary = df.groupby('customer_segment', as_index=False).agg(order_rows=('order_id','count'), net_sales=('net_sales','sum'), avg_net_sales=('net_sales','mean')).sort_values('net_sales', ascending=False)
monthly = df.groupby('order_month', as_index=False).agg(order_rows=('order_id','count'), net_sales=('net_sales','sum'), units_sold=('quantity','sum')).sort_values('order_month')
display(category_summary); display(city_summary); display(segment_summary); display(monthly)


## 3. Correlation matrix

In [ ]:
numeric = df[['quantity','unit_price','discount_pct','gross_sales','discount_amount','net_sales']]
corr = numeric.corr(numeric_only=True)
display(corr.round(3))
fig, ax = plt.subplots(figsize=(8,6))
img = ax.imshow(corr, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
fig.colorbar(img, ax=ax, label='Pearson correlation')
ax.set_title('Numeric Feature Correlation Matrix')
fig.tight_layout(); plt.show()


## 4. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
category_summary.plot(kind='bar', x='category', y='net_sales', ax=ax, legend=False)
ax.set_title('Net Sales by Category'); ax.set_ylabel('Net sales'); ax.set_xlabel('Category')
plt.xticks(rotation=25, ha='right'); fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10,5))
monthly.plot(kind='line', x='order_month', y='net_sales', marker='o', ax=ax, legend=False)
ax.set_title('Monthly Net Sales Trend'); ax.set_ylabel('Net sales'); ax.set_xlabel('Month')
plt.xticks(rotation=60, ha='right'); fig.tight_layout(); plt.show()

category_names = list(df['category'].dropna().unique())
box_data = [df.loc[df['category'].eq(c), 'net_sales'].dropna().values for c in category_names]
fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot(box_data, labels=category_names)
ax.set_title('Net Sales Distribution by Category'); ax.set_ylabel('Net sales')
plt.xticks(rotation=25, ha='right'); fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df['discount_pct'], df['net_sales'], alpha=.25)
ax.set_title('Discount Percentage vs Net Sales'); ax.set_xlabel('Discount (%)'); ax.set_ylabel('Net sales')
fig.tight_layout(); plt.show()


## 5. Hypothesis tests (alpha = 0.05)

H1: Mean net sales differ between Paid and non-Paid orders — Welch t-test.

H2: Discount percentage is associated with net sales — Pearson correlation.

H3: Mean net sales differ across product categories — one-way ANOVA.

Treat these as exploratory tests. Statistical significance is not proof of causation.

In [ ]:
paid = df.loc[df['payment_status'].eq('Paid'), 'net_sales'].dropna()
non_paid = df.loc[~df['payment_status'].eq('Paid'), 'net_sales'].dropna()
t_stat, p_t = stats.ttest_ind(paid, non_paid, equal_var=False)
r, p_corr = stats.pearsonr(df['discount_pct'], df['net_sales'])
category_groups = [g['net_sales'].dropna().values for _, g in df.groupby('category')]
f_stat, p_anova = stats.f_oneway(*category_groups)
hypothesis_results = pd.DataFrame([
 {'hypothesis':'H1: Paid vs non-Paid mean net sales differ','test':'Welch independent-samples t-test','statistic':t_stat,'p_value':p_t,'interpretation':'Significant at 0.05' if p_t < .05 else 'Not significant at 0.05'},
 {'hypothesis':'H2: Discount percentage is associated with net sales','test':'Pearson correlation','statistic':r,'p_value':p_corr,'interpretation':'Significant at 0.05' if p_corr < .05 else 'Not significant at 0.05'},
 {'hypothesis':'H3: Mean net sales differ across categories','test':'One-way ANOVA','statistic':f_stat,'p_value':p_anova,'interpretation':'Significant at 0.05' if p_anova < .05 else 'Not significant at 0.05'}
])
display(hypothesis_results)


## 6. Top five findings and exports

Findings below are generated from the current cleaned dataset, so they update if the dataset changes.

In [ ]:
top_category = category_summary.iloc[0]
top_city = city_summary.iloc[0]
top_month = monthly.loc[monthly['net_sales'].idxmax()]
top_segment = segment_summary.iloc[0]
findings = [
 f"{top_category['category']} leads category net sales at {top_category['net_sales']:,.2f}.",
 f"{top_city['city']} leads city net sales at {top_city['net_sales']:,.2f}.",
 f"{top_month['order_month']} is the highest-net-sales month at {top_month['net_sales']:,.2f}.",
 f"{top_segment['customer_segment']} contributes the most net sales at {top_segment['net_sales']:,.2f}.",
 f"Discount/net-sales Pearson r={r:.3f}, p={p_corr:.3g}; association does not establish causation."
]
for i, finding in enumerate(findings, 1): print(f'{i}. {finding}')
category_summary.to_csv('category_summary.csv', index=False)
city_summary.to_csv('city_summary.csv', index=False)
segment_summary.to_csv('segment_summary.csv', index=False)
monthly.to_csv('monthly_trends.csv', index=False)
corr.to_csv('correlation_matrix.csv')
hypothesis_results.to_csv('hypothesis_tests.csv', index=False)
pd.DataFrame({'finding': findings}).to_csv('top_5_findings.csv', index=False)
print('Exported 7 CSV report files.')
